# VectorCraft Scripting

Example notebook for the Python bridge to VectorCraft's headless CLI (`vectorcraft-cli`): place Plotly/SVG figures into a document and export PDF/SVG/`.vectorcraft`.

Requires VectorCraft installed (Windows default: `C:\\Program Files\\VectorCraft\\vectorcraft-cli.exe`).

In [ ]:
%config IPCompleter.use_jedi = False
%pdb off
%load_ext autoreload
%autoreload 3

import sys
from copy import deepcopy
from typing import Dict, List, Tuple, Optional, Callable, Union, Any
from collections import defaultdict
from functools import partial
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 100)

import IPython
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

from pyphocorehelpers.programming_helpers import IPythonHelpers
from pyphocorehelpers.assertion_helpers import Assert
from pyphocorehelpers.print_helpers import print_keys_if_possible, print_value_overview_only
from pyphocorehelpers.preferences_helpers import set_pho_preferences
from pyphocorehelpers.Filesystem.path_helpers import find_first_extant_path, file_uri_from_path, sanitize_filename_for_Windows
from pyphocorehelpers.gui.Jupyter.simple_widgets import fullwidth_path_widget
from pyphocorehelpers.pho_jupyter_preview_widget.display_helpers import path_repr_with_filesystem_link

set_pho_preferences(is_concise=False)

_notebook_path: Path = Path(IPythonHelpers.try_find_notebook_filepath(IPython.extract_module_locals())).resolve()
_notebook_dir: Path = _notebook_path.parent
_package_root: Path = _notebook_dir.parent
_package_src: Path = _package_root.joinpath('src').resolve()
if str(_package_src) not in sys.path:
    sys.path.insert(0, str(_package_src))

ip = IPython.get_ipython()
ip = path_repr_with_filesystem_link(ip=ip)

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
template: str = 'plotly_dark'
pio.templates.default = template

from vectorcraft_scripting import (
    CliNotFoundError,
    PlaceSpec,
    VectorCraftClient,
    VectorCraftError,
    export_plotly_figure,
    place_into_template,
    write_plotly_svg,
)

print(f'_notebook_path: {_notebook_path}')
print(f'_package_src: {_package_src}')

## Paths / output folder

In [ ]:
outputs_folder: Path = _notebook_dir.joinpath('outputs').resolve()
outputs_folder.mkdir(parents=True, exist_ok=True)
display(fullwidth_path_widget(outputs_folder, file_name_label='VectorCraft outputs:'))
print(f'outputs_folder: {outputs_folder}')
print(f'file_uri: {file_uri_from_path(outputs_folder)}')

## Discover `vectorcraft-cli` (Windows)

In [ ]:
vc = VectorCraftClient()  # VECTORCRAFT_CLI → PATH → C:\\Program Files\\VectorCraft\\vectorcraft-cli.exe
print(f'cli_path: {vc.cli_path}')
Assert.path_exists(vc.cli_path)

catalogue: List[Dict[str, Any]] = vc.commands()
print(f'n_commands: {len(catalogue)}')
print(f'first: {catalogue[0]["id"]}')

place_docs = next((c for c in catalogue if c.get('id') == 'file.place'), None)
if place_docs is not None:
    print(f"file.place enabled={place_docs.get('enabled')} label={place_docs.get('label')!r}")
    print(str(place_docs.get('params', ''))[:500], '...')

## Smoke: `file.new` → export SVG

In [ ]:
smoke_svg: Path = outputs_folder.joinpath('smoke_blank.svg').resolve()
smoke_steps = vc.run(
    commands=[('file.new', {'width': 560, 'height': 360})],
    exports=[smoke_svg],
)
print_value_overview_only(smoke_steps)
Assert.path_exists(smoke_svg)
display(fullwidth_path_widget(smoke_svg, file_name_label='smoke SVG:'))

## Plotly figure → SVG → place → export

Creates a new document, places the chart in a rect (document points: `[x, y, width, height]` from the artboard top-left), and writes `.vectorcraft` + PDF.

In [ ]:
fig = go.Figure(go.Scatter(x=[1, 2, 3], y=[2, 1, 3], mode='lines+markers', name='demo'))
fig.update_layout(width=560, height=360, margin=dict(l=40, r=16, t=32, b=32), title='VectorCraft place demo')
fig.show()

chart_svg: Path = outputs_folder.joinpath('chart.svg').resolve()
write_plotly_svg(fig, chart_svg)
Assert.path_exists(chart_svg)

# 36 pt from left, 80 pt from top, 280×180 pt box
place = PlaceSpec(path=chart_svg, rect=[36, 80, 280, 180], crop='crop', link=False)

out_vc: Path = outputs_folder.joinpath('out.vectorcraft').resolve()
out_pdf: Path = outputs_folder.joinpath('out.pdf').resolve()

place_steps = vc.run(
    commands=[
        ('file.new', {'width': 612, 'height': 792}),  # letter-ish artboard
        place.as_command(),
    ],
    exports=[out_vc, out_pdf],
)
print_keys_if_possible('place_steps', place_steps, max_depth=2)
Assert.path_exists(out_vc)
Assert.path_exists(out_pdf)
display(fullwidth_path_widget(out_pdf, file_name_label='placed PDF:'))

## Optional: place into an existing `.vectorcraft` template

If you have a layout template, set `template_path` and run the cell below (same as the README `export_plotly_figure` / `place_into_template` helpers).

In [ ]:
template_path: Optional[Path] = None
# template_path = Path(r'H:/path/to/template.vectorcraft').resolve()

if template_path is None:
    print('Set template_path to a .vectorcraft file to run this cell.')
else:
    Assert.path_exists(template_path)
    templated_pdf: Path = outputs_folder.joinpath('from_template.pdf').resolve()
    results = export_plotly_figure(
        fig,
        chart_svg=outputs_folder.joinpath('chart_for_template.svg'),
        template=template_path,
        rect=[36, 480, 280, 180],
        exports=[templated_pdf],
        client=vc,
    )
    print_value_overview_only(results)
    display(fullwidth_path_widget(templated_pdf, file_name_label='template PDF:'))

## Inspect a document

In [ ]:
summary = vc.info(out_vc)
print_keys_if_possible('info', summary, max_depth=2)
summary.get('artboards')

## Live session (`vectorcraft --control`)

Requires the desktop app started with a control port, e.g. `vectorcraft --control 7979`. Discovery does not launch the GUI.

In [ ]:
from vectorcraft_scripting import ObjectQuery, connect_first, discover_instances, NoInstanceError, LiveVectorCraft

instances = discover_instances()
print(f'n_instances: {len(instances)}')
for inst in instances:
    print(f'  port={inst.port} pid={inst.pid}')
## END for inst in instances...

live: Optional[LiveVectorCraft] = None
try:
    live = connect_first()
    docs = live.list_documents()
    print('documents:', [d.get('title') for d in docs])
    tree = live.inspect_document(depth=2, child_limit=50)
    print_keys_if_possible('inspect_document', tree, max_depth=2)
    matches = live.find_objects(ObjectQuery(kind='Path', min_width=10, max_width=2000, limit=25))
    print(f'Path matches: {len(matches)}')
    if matches:
        print(matches[0])
except NoInstanceError as e:
    print(f'No live VectorCraft control server: {e}')
    print('Start with: & "C:\\Program Files\\VectorCraft\\vectorcraft.exe" --control 7979')
# finally:
#     if live is not None:
#         live.close()

In [ ]:
live.find_objects(

In [ ]:
active_document = docs[0]
active_document

active_artboard = active_document['artboards'][0] ##
active_artboard

In [ ]:
# print('documents:', [d.get('title') for d in docs])
tree = live.inspect_document(depth=3, child_limit=100)
# print_keys_if_possible('inspect_document', tree, max_depth=2)

layers = tree['layers']
layers


In [ ]:
layer_names = ['kdiba_4_figure4_summary_four_bar-Laps.vectorcraft', 'ScatterOverTime_Laps_perTimeBin.svg', 'ScatterOverTime_PBEs_perTimeBin.svg', 'kdiba_4_figure4_summary_four_bar-PBEs.vectorcraft']


In [ ]:

print_keys_if_possible('layers', layers, max_depth=0)


In [ ]:

matches = live.find_objects(ObjectQuery(kind='Path', min_width=10, max_width=2000, limit=25))
print(f'Path matches: {len(matches)}')
if matches:
    print(matches[0])



In [ ]:

matches = live.find_objects(ObjectQuery(kind='Path', min_width=10, max_width=2000, limit=25))
print(f'Path matches: {len(matches)}')
if matches:
    print(matches[0])


In [ ]:
from vectorcraft_scripting import connect_first, ObjectQuery

live = connect_first()
print(live.list_documents())
print(live.find_objects(ObjectQuery(kind="Path", min_width=100)))
live.close()

## Get specific layers

In [ ]:

def get_layer_geometry(live, layer_names: List[str]) -> pd.DataFrame:
    def _subfn_bounds_as_xywh(bounds) -> Optional[Dict[str, float]]:
        if isinstance(bounds, dict):
            return dict(x=float(bounds['x']), y=float(bounds['y']), width=float(bounds['width']), height=float(bounds['height']))
        if isinstance(bounds, (list, tuple)) and len(bounds) >= 4:
            return dict(x=float(bounds[0]), y=float(bounds[1]), width=float(bounds[2]), height=float(bounds[3]))
        return None


    def _subfn_flatten_layer_nodes(nodes: List[Any]) -> List[Dict[str, Any]]:
        out: List[Dict[str, Any]] = []
        for node in nodes:
            if not isinstance(node, dict):
                continue
            out.append(node)
            children = node.get('children') or node.get('layers') or []
            if isinstance(children, list) and children:
                out.extend(_subfn_flatten_layer_nodes(children))
        ## END for node in nodes...
        return out


    def _subfn_node_geom_row(query_name: str, node: Dict[str, Any]) -> Dict[str, Any]:
        xywh = _subfn_bounds_as_xywh(node.get('bounds')) or {}
        return {'query': query_name, 'id': node.get('id'), 'name': node.get('name'), 'kind': node.get('kind'), 'x': xywh.get('x'), 'y': xywh.get('y'), 'width': xywh.get('width'), 'height': xywh.get('height'), 'visible': node.get('visible'), 'locked': node.get('locked')}


    if live is None:
        live = connect_first()

    info = live.active_document_info()
    print(f"active document: {info.get('title')!r}  units={info.get('units')!r}")

    tree = live.inspect_document(depth=8, child_limit=500)
    flat_nodes = _subfn_flatten_layer_nodes(list(tree.get('layers') or []))

    rows: List[Dict[str, Any]] = []
    for name in layer_names:
        matches = live.find_objects(ObjectQuery(name=name, limit=50))
        match_ids = [m['id'] for m in matches if isinstance(m, dict) and 'id' in m]
        found: List[Dict[str, Any]] = []
        if match_ids:
            inspected = live.inspect_document(ids=match_ids, depth=0)
            found = list(inspected.get('nodes') or []) if isinstance(inspected, dict) else []
            if not found:
                found = matches
        if not found:
            found = [n for n in flat_nodes if name in str(n.get('name') or '')]
        if not found:
            rows.append(_subfn_node_geom_row(name, {}))
            continue
        seen_ids = set()
        for node in found:
            node_id = node.get('id')
            if node_id in seen_ids:
                continue
            if node_id is not None:
                seen_ids.add(node_id)
            if (node.get('bounds') is None) and (node_id is not None):
                node = live.get_node(node_id, summary=True, depth=0) or node
            rows.append(_subfn_node_geom_row(name, node))
        ## END for node in found...
    ## END for name in layer_names...

    layer_geom_df = pd.DataFrame(rows)
    layer_geom_df.set_index('name', drop=True, inplace=True)
    return layer_geom_df

# layer_names = ['kdiba_4_figure4_summary_four_bar-Laps.vectorcraft', 'ScatterOverTime_Laps_perTimeBin.svg', 'ScatterOverTime_PBEs_perTimeBin.svg', 'kdiba_4_figure4_summary_four_bar-PBEs.vectorcraft']

layer_names = ['ScatterOverTime_Laps_perTimeBin.svg','ScatterOverTime_PBEs_perTimeBin.svg','kdiba_4_Figure S8-pre_post_delta_effect-P_Short.svg']
layer_geom_df = get_layer_geometry(live=live, layer_names=layer_names)
layer_geom_df

geom_columns = ['x', 'y', 'width', 'height'] # 'name', 
curr_fig_geom_dict = layer_geom_df[geom_columns].to_dict(orient='index')
## OUTPUTS: curr_fig_geom_dict


In [ ]:
figs_layer_geometry = {
    'Artboard 4.pdf': {
        'kdiba_4_figure4_summary_four_bar-Laps.vectorcraft': {'x': 369.36756389662094, 'y': -13.21314402582275, 'width': 110.88246366869805, 'height': 166.32371202582274},
        'ScatterOverTime_Laps_perTimeBin.svg': {'x': 0.7921446789572979, 'y': 0.0, 'width': 388.47986937169003, 'height': 156.96156338250103},
        'ScatterOverTime_PBEs_perTimeBin.svg': {'x': 0.5088833995630582, 'y': 150.66224712966167, 'width': 388.47986937169, 'height': 156.961563382501},
        'kdiba_4_figure4_summary_four_bar-PBEs.vectorcraft': {'x': 370.11565144042964, 'y': 141.04640021926545, 'width': 112.38492903082448, 'height': 168.5774102928972}
    },

    'Figure S8 - Effect of Heuristic-filtering perTimeBin.pdf': {
        'ScatterOverTime_Laps_perTimeBin.svg': {'x': 0.0, 'y': 0.0, 'width': 468.27133896419633, 'height': 189.20054099563487},
        'ScatterOverTime_PBEs_perTimeBin.svg': {'x': 0.5893954560219754, 'y': 185.72098141999123, 'width': 467.69220359180815, 'height': 188.96654690578106},
        'kdiba_4_Figure S8-pre_post_delta_effect-P_Short.svg': {'x': 6.4321670348433475, 'y': 364.941057936045, 'width': 278.90913365273093, 'height': 209.8713396547812},
    },

}

replacement_paths_dict = {
    'Artboard 4.pdf': Path('E:/Dropbox (Personal)/Active/Kamran Diba Lab/Pho-Kamran-Meetings/Pho Diba Paper 2023/Figures/UneditedSources/Figure 4 - P_Short Scatter over Time/2026-10-29'),
    'Figure S8 - Effect of Heuristic-filtering perTimeBin.pdf': Path(r'E:\Dropbox (Personal)\Active\Kamran Diba Lab\Pho-Kamran-Meetings\Pho Diba Paper 2023\Figures\UneditedSources\Figure S8 - Effect of Heuristic-filtering P_Short Scatter over Time - perEpoch\Inputs\2026-10-29'),
}


## make a complete backup of the document to the document's file path + "_BAK"
#TODO 2026-10-29 09:11: - [ ] BACKUP THE DOCUMENT

## For a particular figure
active_figure_name: str = 'Figure S8 - Effect of Heuristic-filtering perTimeBin.pdf'

## search for files with names matching those in figs_layer_geometry in `curr_replacement_search_path`
# for active_figure_name, curr_fig_geom_dict in figs_layer_geometry.items():
curr_fig_geom_dict = figs_layer_geometry[active_figure_name]
curr_replacement_search_path: Path = replacement_paths_dict[active_figure_name]

for a_layer_name, a_layer_geom in curr_fig_geom_dict.items():
    replacement_file: Path = curr_replacement_search_path.joinpath(a_layer_name)
    assert replacement_file.exists(), f"replacement_file: '{replacement_file.as_posix()}' does not exist!"
    ## back up the existing layer to `a_layer_name`
    backup_layer_name: str = f"{a_layer_name}_{TODAY_DAY_DATE}" ## incrementing as needed
    #TODO 2026-10-29 09:08: - [ ] Copy the layer

    ## insert the `replacement_file` at exactly the dimensions specified in `a_layer_geom`, naming the new layer `new_layer_name`
    new_layer_name: str = f"{a_layer_name}_PROGRAMMATIC"
    #TODO 2026-10-29 09:09: - [ ] PERFORM INSERT WITH CORRECT SIZE, WARNING IF NON-UNIFORM SCALING WOULD BE REQUIRED to position the image there



In [ ]:
import time
from pathlib import Path
from typing import Dict, List, Optional, Tuple

from vectorcraft_scripting import LiveVectorCraft

if 'TODAY_DAY_DATE' not in globals():
    from datetime import datetime
    TODAY_DAY_DATE: str = datetime.now().strftime('%Y-%m-%d')


def active_document_path(live: LiveVectorCraft) -> Path:
    """On-disk path of the active document (from `document.inspect`'s `path`)."""
    tree = live.inspect_document(depth=0, childLimit=0)
    doc_path: str = str(tree.get('path') or '')
    assert doc_path, f"active document {tree.get('title')!r} has never been saved (no path) — save it first, then back it up"
    return Path(doc_path)


def unique_backup_path(backup_dir: Path, stem: str, date_str: str, suffix: str) -> Path:
    """`<stem>_<date><suffix>`, else `<stem>_<date>_1<suffix>`, `_2`, … — never overwrites an existing backup."""
    candidate: Path = backup_dir.joinpath(f"{stem}_{date_str}{suffix}")
    i: int = 0
    while candidate.exists():
        i += 1
        candidate = backup_dir.joinpath(f"{stem}_{date_str}_{i}{suffix}")
    return candidate


def wait_for_background_jobs(live: LiveVectorCraft, timeout_sec: float = 120.0, poll_sec: float = 0.25) -> None:
    """Block until the desktop app's Background Save/Export jobs (`ui.inspect` → `background`) are done."""
    t_start: float = time.monotonic()
    while True:
        ui = live.ui_inspect()
        jobs: List = list(ui.get('background') or []) if isinstance(ui, dict) else []
        if not jobs:
            return
        assert (time.monotonic() - t_start) < timeout_sec, f"background jobs still running after {timeout_sec}s: {jobs}"
        time.sleep(poll_sec)


def save_document_copy(live: LiveVectorCraft, out_path: Path, timeout_sec: float = 120.0) -> Dict:
    """`file.saveCopy {path}` → the document keeps its own path/title/modified state. Format comes from `out_path`'s extension."""
    reply = live.execute('file.saveCopy', {'path': str(out_path.resolve())})
    assert reply != 'cancelled', f"file.saveCopy was cancelled (a dialog was opened) for '{out_path.as_posix()}' — reply={reply!r}"
    if isinstance(reply, dict) and reply.get('pending'):
        raise RuntimeError(f"file.saveCopy opened a {reply['pending']!r} dialog instead of writing '{out_path.as_posix()}'; answer it in the app or change the backup format")
    if isinstance(reply, dict) and reply.get('background'):
        wait_for_background_jobs(live, timeout_sec=timeout_sec)
    assert out_path.exists(), f"backup not written: '{out_path.as_posix()}' (reply={reply!r})"
    if isinstance(reply, dict) and reply.get('warnings'):
        print(f"  warnings for '{out_path.name}': {reply['warnings']}")
    return reply if isinstance(reply, dict) else {'result': reply}


def backup_active_document(live: LiveVectorCraft, date_str: str = TODAY_DAY_DATE, backup_subfolder: str = '_BAK', also_native_copy: bool = True) -> Dict[str, Path]:
    """Back up the active document into `<doc folder>/_BAK/<stem>_<date><ext>` (creating `_BAK` if needed).

    `also_native_copy=True` additionally writes a lossless `<stem>_<date>.vectorcraft` next to it, since a PDF/SVG-format
    copy of a PDF/SVG-backed document re-exports through that format and can lose VectorCraft-only state.
    """
    doc_path: Path = active_document_path(live)
    assert doc_path.exists(), f"document path reported by VectorCraft does not exist on disk: '{doc_path.as_posix()}'"
    backup_dir: Path = doc_path.parent.joinpath(backup_subfolder)
    backup_dir.mkdir(parents=True, exist_ok=True)

    targets: List[Tuple[str, Path]] = [('same_format', unique_backup_path(backup_dir, doc_path.stem, date_str, doc_path.suffix))]
    if also_native_copy and (doc_path.suffix.lower() != '.vectorcraft'):
        targets.append(('vectorcraft', unique_backup_path(backup_dir, doc_path.stem, date_str, '.vectorcraft')))

    out: Dict[str, Path] = {}
    print(f"backing up {doc_path.name!r} → '{backup_dir.as_posix()}'")
    for label, out_path in targets:
        save_document_copy(live, out_path)
        print(f"  [{label}] {out_path.as_posix()}  ({out_path.stat().st_size / 1024:.1f} KiB)")
        out[label] = out_path
    ## END for label, out_path in targets...

    return out

In [ ]:
import json
import math
import re
import warnings
from datetime import datetime
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import pandas as pd
from vectorcraft_scripting import ControlError, LiveVectorCraft, ObjectQuery, connect_first

if 'TODAY_DAY_DATE' not in globals():
    TODAY_DAY_DATE: str = datetime.now().strftime('%Y-%m-%d')

figs_layer_geometry = {
    'Artboard 4.pdf': {
        'kdiba_4_figure4_summary_four_bar-Laps.vectorcraft': {'x': 369.36756389662094, 'y': -13.21314402582275, 'width': 110.88246366869805, 'height': 166.32371202582274},
        'ScatterOverTime_Laps_perTimeBin.svg': {'x': 0.7921446789572979, 'y': 0.0, 'width': 388.47986937169003, 'height': 156.96156338250103},
        'ScatterOverTime_PBEs_perTimeBin.svg': {'x': 0.5088833995630582, 'y': 150.66224712966167, 'width': 388.47986937169, 'height': 156.961563382501},
        'kdiba_4_figure4_summary_four_bar-PBEs.vectorcraft': {'x': 370.11565144042964, 'y': 141.04640021926545, 'width': 112.38492903082448, 'height': 168.5774102928972}
    },

    'Figure S8 - Effect of Heuristic-filtering perTimeBin.pdf': {
        'ScatterOverTime_Laps_perTimeBin.svg': {'x': 0.0, 'y': 0.0, 'width': 468.27133896419633, 'height': 189.20054099563487},
        'ScatterOverTime_PBEs_perTimeBin.svg': {'x': 0.5893954560219754, 'y': 185.72098141999123, 'width': 467.69220359180815, 'height': 188.96654690578106},
        'kdiba_4_Figure S8-pre_post_delta_effect-P_Short.svg': {'x': 6.4321670348433475, 'y': 364.941057936045, 'width': 278.90913365273093, 'height': 209.8713396547812},
    },

}

replacement_paths_dict = {
    'Artboard 4.pdf': Path('E:/Dropbox (Personal)/Active/Kamran Diba Lab/Pho-Kamran-Meetings/Pho Diba Paper 2023/Figures/UneditedSources/Figure 4 - P_Short Scatter over Time/2026-10-29'),
    'Figure S8 - Effect of Heuristic-filtering perTimeBin.pdf': Path(r'E:\Dropbox (Personal)\Active\Kamran Diba Lab\Pho-Kamran-Meetings\Pho Diba Paper 2023\Figures\UneditedSources\Figure S8 - Effect of Heuristic-filtering P_Short Scatter over Time - perEpoch\Inputs\2026-10-29'),
}


# ---------------------------------------------------------------------------------------------------------------- #
# Helpers
# ---------------------------------------------------------------------------------------------------------------- #
_SVG_LENGTH_TO_PT: Dict[str, float] = {'': 1.0, 'px': 1.0, 'pt': 1.0, 'in': 72.0, 'cm': 72.0 / 2.54, 'mm': 72.0 / 25.4, 'pc': 12.0}


def _subfn_svg_length_to_pt(value: str) -> Optional[float]:
    m = re.fullmatch(r'\s*([0-9.]+(?:e[-+]?\d+)?)\s*([a-z%]*)\s*', value, flags=re.IGNORECASE)
    if m is None or m.group(2).lower() == '%':
        return None
    return float(m.group(1)) * _SVG_LENGTH_TO_PT[m.group(2).lower()]


def source_file_size_pts(path: Path) -> Optional[Tuple[float, float]]:
    """Intrinsic (width, height) in document points of a replacement file, or None if unknown (e.g. PDF)."""
    suffix: str = path.suffix.lower()
    if suffix == '.svg':
        head: str = path.read_text(encoding='utf-8', errors='replace')[:4096]
        svg_tag = re.search(r'<svg\b[^>]*>', head, flags=re.IGNORECASE | re.DOTALL)
        assert svg_tag is not None, f"no <svg> tag found in '{path.as_posix()}'"
        tag: str = svg_tag.group(0)
        w_attr = re.search(r'\bwidth="([^"]+)"', tag)
        h_attr = re.search(r'\bheight="([^"]+)"', tag)
        if (w_attr is not None) and (h_attr is not None):
            w_pt, h_pt = _subfn_svg_length_to_pt(w_attr.group(1)), _subfn_svg_length_to_pt(h_attr.group(1))
            if (w_pt is not None) and (h_pt is not None):
                return (w_pt, h_pt)
        vb = re.search(r'\bviewBox="([^"]+)"', tag)
        if vb is not None:
            _vx, _vy, vw, vh = [float(v) for v in re.split(r'[\s,]+', vb.group(1).strip())]
            return (vw, vh)
        return None
    if suffix == '.vectorcraft':
        doc = json.loads(path.read_text(encoding='utf-8'))
        rect = doc['document']['artboards'][0]['rect']
        return (float(rect['x1']) - float(rect['x0']), float(rect['y1']) - float(rect['y0']))
    return None


def uniform_fit_rect(target: Dict[str, float], source_size: Tuple[float, float]) -> Dict[str, float]:
    """Largest rect with the source aspect ratio that fits inside `target`, centered in it."""
    sw, sh = source_size
    scale: float = min(target['width'] / sw, target['height'] / sh)
    w, h = sw * scale, sh * scale
    return {'x': target['x'] + (target['width'] - w) / 2.0, 'y': target['y'] + (target['height'] - h) / 2.0, 'width': w, 'height': h}


def find_layer_nodes(live: LiveVectorCraft, name: str, exact: bool = True) -> List[Dict[str, Any]]:
    """`document.find` by Layers-panel name (substring match server-side; exact filter client-side)."""
    result = live.execute('document.find', {'name': name, 'limit': 100})
    matches: List[Dict[str, Any]] = list(result.get('matches') or []) if isinstance(result, dict) else list(result or [])
    if exact:
        exact_matches = [m for m in matches if str(m.get('name') or '') == name]
        if exact_matches:
            return exact_matches
    return matches


def node_bounds(live: LiveVectorCraft, node_id: Any) -> Dict[str, float]:
    node = live.get_node(node_id, summary=True, depth=0)
    b = node['bounds']
    return {'x': float(b['x']), 'y': float(b['y']), 'width': float(b['width']), 'height': float(b['height'])}


def unique_backup_name(live: LiveVectorCraft, base_name: str) -> str:
    """`base_name`, else `base_name_1`, `base_name_2`, … until no layer/object has that exact name."""
    candidate: str = base_name
    i: int = 0
    while find_layer_nodes(live, candidate, exact=True):
        i += 1
        candidate = f"{base_name}_{i}"
    return candidate


def resolve_duplicate_command(live: LiveVectorCraft) -> Optional[str]:
    """First duplicate-like command id present in the live catalogue (None → fall back to `object.move copy`)."""
    catalogue = live.connection.engine_commands()
    ids = {c.get('id') for c in catalogue if isinstance(c, dict)}
    for candidate in ('layer.duplicate', 'object.duplicate', 'edit.duplicate'):
        if candidate in ids:
            return candidate
    ## END for candidate in ('layer.duplicate', 'object.duplicate', 'edit.duplicate')...
    return None


def duplicate_nodes(live: LiveVectorCraft, ids: List[Any], duplicate_command: Optional[str]) -> List[Any]:
    """Duplicate `ids` in place and return the new ids (whatever the reply doesn't name is read from the selection)."""
    if duplicate_command is not None:
        reply = live.execute(duplicate_command, {'ids': list(ids)})
    else:
        reply = live.execute('object.move', {'ids': list(ids), 'dx': 0.0, 'dy': 0.0, 'copy': True})
    new_ids: List[Any] = []
    if isinstance(reply, dict):
        new_ids = list(reply.get('ids') or ([reply['id']] if 'id' in reply else []))
    if not new_ids:
        tree = live.inspect_document(depth=0, childLimit=0)
        new_ids = [s for s in (tree.get('selection') or []) if s not in ids]
    assert new_ids, f"duplicate of ids={ids} produced no new ids (reply={reply!r})"
    return new_ids


def place_replacement(live: LiveVectorCraft, replacement_file: Path, rect: Dict[str, float], new_layer_name: str, link: bool = False) -> Tuple[List[Any], Dict[str, float]]:
    """`file.place` at `rect` ([x, y, w, h] document points, artboard top-left), rename, and read back the actual bounds."""
    params: Dict[str, Any] = {'path': str(replacement_file.resolve()), 'rect': [rect['x'], rect['y'], rect['width'], rect['height']], 'crop': 'crop', 'link': link}
    reply = live.execute('file.place', params)
    placed_ids: List[Any] = []
    if isinstance(reply, dict):
        placed_ids = list(reply.get('ids') or ([reply['id']] if 'id' in reply else []))
    if not placed_ids:
        tree = live.inspect_document(depth=0, childLimit=0)
        placed_ids = list(tree.get('selection') or [])
    assert placed_ids, f"file.place returned no ids for '{replacement_file.as_posix()}' (reply={reply!r})"
    live.execute('object.setProps', {'ids': placed_ids, 'name': new_layer_name})
    actual = node_bounds(live, placed_ids[0])
    return placed_ids, actual


def replace_figure_layers(live: LiveVectorCraft, active_figure_name: str, curr_fig_geom_dict: Dict[str, Dict[str, float]], curr_replacement_search_path: Path,
                          backup_suffix: str = TODAY_DAY_DATE, new_suffix: str = '_PROGRAMMATIC', hide_original: bool = True, hide_backup: bool = True,
                          allow_non_uniform_scaling: bool = False, aspect_tolerance: float = 0.01, link: bool = False) -> pd.DataFrame:
    """For each layer in `curr_fig_geom_dict`: duplicate the existing layer as a (hidden) backup, place the replacement file at the stored geometry, and report actual vs. target bounds."""
    info = live.active_document_info()
    assert info['title'] == active_figure_name, f"active document is {info['title']!r}, expected {active_figure_name!r} — activate the right tab first (live.list_documents())"
    print(f"active document: {info['title']!r}  units={info['units']!r}")

    duplicate_command: Optional[str] = resolve_duplicate_command(live)
    print(f"duplicate command: {duplicate_command or 'object.move {copy: true}'}")

    rows: List[Dict[str, Any]] = []
    for a_layer_name, a_layer_geom in curr_fig_geom_dict.items():
        replacement_file: Path = curr_replacement_search_path.joinpath(a_layer_name)
        assert replacement_file.exists(), f"replacement_file: '{replacement_file.as_posix()}' does not exist!"

        ## existing layer(s) with this exact name
        existing = find_layer_nodes(live, a_layer_name, exact=True)
        existing_ids: List[Any] = [m['id'] for m in existing]
        if not existing_ids:
            warnings.warn(f"no existing layer named {a_layer_name!r} in {active_figure_name!r}; placing without backup")

        ## back up the existing layer to `backup_layer_name` (incrementing as needed)
        backup_layer_name: Optional[str] = None
        backup_ids: List[Any] = []
        if existing_ids:
            backup_layer_name = unique_backup_name(live, f"{a_layer_name}_{backup_suffix}")
            backup_ids = duplicate_nodes(live, existing_ids, duplicate_command)
            live.execute('object.setProps', {'ids': backup_ids, 'name': backup_layer_name, 'visible': (not hide_backup)})
            if hide_original:
                live.execute('object.setProps', {'ids': existing_ids, 'visible': False})

        ## aspect check: warn if placing at exactly `a_layer_geom` would require non-uniform scaling
        source_size = source_file_size_pts(replacement_file)
        target_rect: Dict[str, float] = dict(a_layer_geom)
        place_rect: Dict[str, float] = dict(a_layer_geom)
        aspect_ratio_error: Optional[float] = None
        if source_size is not None:
            target_aspect: float = target_rect['width'] / target_rect['height']
            source_aspect: float = source_size[0] / source_size[1]
            aspect_ratio_error = abs(math.log(target_aspect / source_aspect))
            if aspect_ratio_error > aspect_tolerance:
                msg = (f"{a_layer_name!r}: target aspect {target_aspect:.4f} ({target_rect['width']:.2f}x{target_rect['height']:.2f} pt) != source aspect {source_aspect:.4f} "
                       f"({source_size[0]:.2f}x{source_size[1]:.2f} pt); non-uniform scaling would be required (|log ratio| = {aspect_ratio_error:.4f}).")
                if allow_non_uniform_scaling:
                    warnings.warn(msg + ' Placing at the exact target rect anyway.')
                else:
                    place_rect = uniform_fit_rect(target_rect, source_size)
                    warnings.warn(msg + f" Placing uniformly-scaled and centered instead: {place_rect}")
        else:
            warnings.warn(f"{a_layer_name!r}: could not read intrinsic size of '{replacement_file.suffix}' file; aspect not checked.")

        ## insert the `replacement_file` at `place_rect`, naming the new layer `new_layer_name`
        new_layer_name: str = f"{a_layer_name}{new_suffix}"
        placed_ids, actual = place_replacement(live, replacement_file, place_rect, new_layer_name, link=link)

        rows.append({'layer': a_layer_name, 'replacement_file': replacement_file.as_posix(), 'existing_ids': existing_ids, 'backup_layer': backup_layer_name, 'backup_ids': backup_ids,
                     'new_layer': new_layer_name, 'placed_ids': placed_ids, 'source_w': (source_size or (None, None))[0], 'source_h': (source_size or (None, None))[1], 'aspect_ratio_error': aspect_ratio_error,
                     'target_x': target_rect['x'], 'target_y': target_rect['y'], 'target_w': target_rect['width'], 'target_h': target_rect['height'],
                     'placed_x': place_rect['x'], 'placed_y': place_rect['y'], 'placed_w': place_rect['width'], 'placed_h': place_rect['height'],
                     'actual_x': actual['x'], 'actual_y': actual['y'], 'actual_w': actual['width'], 'actual_h': actual['height'],
                     'dx': actual['x'] - place_rect['x'], 'dy': actual['y'] - place_rect['y'], 'dw': actual['width'] - place_rect['width'], 'dh': actual['height'] - place_rect['height']})
    ## END for a_layer_name, a_layer_geom in curr_fig_geom_dict.items()...

    return pd.DataFrame(rows).set_index('layer', drop=True)


# ---------------------------------------------------------------------------------------------------------------- #
# Run — for a particular figure
# ---------------------------------------------------------------------------------------------------------------- #
active_figure_name: str = 'Figure S8 - Effect of Heuristic-filtering perTimeBin.pdf'
curr_fig_geom_dict = figs_layer_geometry[active_figure_name]
curr_replacement_search_path: Path = replacement_paths_dict[active_figure_name]

if ('live' not in globals()) or (live is None):
    live = connect_first()


backup_paths: Dict[str, Path] = backup_active_document(live)   # e.g. {'same_format': .../_BAK/Artboard 4_2026-10-29.pdf, 'vectorcraft': .../_BAK/Artboard 4_2026-10-29.vectorcraft}
print(f'backup_paths: "{backup_paths}"')
## one undo step for the whole replacement
live.execute('edit.undoGroup.begin', {'label': f'Replace figure layers ({TODAY_DAY_DATE})'})
try:
    replacement_df = replace_figure_layers(live=live, active_figure_name=active_figure_name, curr_fig_geom_dict=curr_fig_geom_dict, curr_replacement_search_path=curr_replacement_search_path,
                                           hide_original=True, hide_backup=True, allow_non_uniform_scaling=False)
finally:
    live.execute('edit.undoGroup.end', {})

replacement_df